# Avro Format Demonstration

## Overview
Avro is a row-oriented remote procedure call and data serialization framework developed within Apache's Hadoop project. It uses JSON for defining data types and protocols, and serializes data in a compact binary format. This notebook demonstrates Avro operations using **PySpark** for distributed data processing.

## Key Characteristics of Avro Format

### 1. **Row-Based Storage**
   - Data is stored row-by-row (unlike columnar formats like ORC/Parquet)
   - Benefits: Fast writes, good for streaming and transactional systems
   - Ideal for write-heavy workloads and full-row access patterns

### 2. **Schema Evolution**
   - Built-in support for schema evolution
   - Readers can use different schema than writers (schema resolution)
   - Backward and forward compatibility support
   - No need to regenerate data when schema changes

### 3. **Binary Serialization**
   - Compact binary encoding
   - Self-describing data (schema embedded in file)
   - No field tags or delimiters in data
   - Efficient serialization/deserialization

### 4. **Language Neutral**
   - Rich data structures
   - Code generation for multiple languages (Java, Python, C++, etc.)
   - Schema defined in JSON
   - Platform-independent binary format

### 5. **Splittable Format**
   - Can be split for parallel processing
   - Good for MapReduce and Spark workloads
   - Each file contains its own schema
   - No need for header files

### 6. **Compression Support**
   - Built-in compression codecs (SNAPPY, DEFLATE, BZIP2)
   - Block-level compression
   - Good compression ratios for structured data

### 7. **Performance Features**
   - Fast serialization/deserialization
   - Smaller file sizes than JSON/XML
   - Efficient for messaging systems (Kafka)
   - Good for data exchange between systems

### 8. **Use Cases**
   - Data serialization in messaging systems (Apache Kafka)
   - RPC (Remote Procedure Call) frameworks
   - Data exchange between different systems
   - Streaming data pipelines
   - ETL intermediate format
   - Event logging and audit trails

In [1]:
# Set environment variables for PySpark
import os
import sys
try:
    import findspark
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "findspark"])
    import findspark

# Set Spark and Java homes using standard Homebrew paths
# For Spark 4.1.1 on Apple Silicon, the real home is in libexec
os.environ['SPARK_HOME'] = '/opt/homebrew/opt/apache-spark/libexec'
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home'
os.environ['PYSPARK_PYTHON'] = sys.executable

# Initialize findspark
findspark.init(os.environ['SPARK_HOME'])

# Suppress verbose logging
import logging
logging.basicConfig(level=logging.ERROR)

# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType, TimestampType
import pyspark.sql.functions as F
from datetime import datetime, timedelta
import pandas as pd

print("Initializing PySpark...")

# Initialize SparkSession with master("local[*]") to prevent hanging
# Note: Avro support requires the spark-avro package
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Avro_Format_Demo") \
    .config("spark.jars.packages", "org.apache.spark:spark-avro_2.12:4.1.1") \
    .config("spark.driver.memory", "1g") \
    .config("spark.executor.memory", "1g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("✓ SparkSession initialized successfully!")
print(f"Spark version: {spark.version}")
print("✓ Avro format support enabled")
print("✓ All operations will use PySpark")

Initializing PySpark...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/20 10:18:22 WARN Utils: Your hostname, mukeshs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.29.221 instead (on interface en0)
26/01/20 10:18:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/mukesh/.ivy2.5.2/cache
The jars for the packages stored in: /Users/mukesh/.ivy2.5.2/jars
org.apache.spark#spark-avro_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8071cc8a-2fda-4c85-8238-19eb4b3b33b9;1.0
	confs: [default]
:: resolution report :: resolve 2908ms :: artifacts dl 0ms
	:: modules in use:
	---------------------------------------------------------------------
	|                  |            modules            ||   artifact

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

## Step 1: Create Sample Dataset (10 rows)

In [ ]:
# Create sample data with 10 rows using PySpark
data = [
    (101, 'Alice Johnson', 'Sales', 65000, '2020-01-15', 3.85, True, 5),
    (102, 'Bob Smith', 'IT', 85000, '2020-02-15', 4.62, True, 8),
    (103, 'Charlie Brown', 'HR', 55000, '2020-03-15', 3.21, False, 3),
    (104, 'Diana Prince', 'Finance', 75000, '2020-04-15', 4.45, True, 6),
    (105, 'Eve Wilson', 'Sales', 68000, '2020-05-15', 3.95, True, 5),
    (106, 'Frank Miller', 'IT', 90000, '2020-06-15', 4.78, True, 10),
    (107, 'Grace Lee', 'HR', 58000, '2020-07-15', 3.12, False, 2),
    (108, 'Henry Davis', 'Finance', 78000, '2020-08-15', 4.35, True, 7),
    (109, 'Iris Anderson', 'Sales', 70000, '2020-09-15', 4.05, True, 6),
    (110, 'Jack Wilson', 'IT', 88000, '2020-10-15', 4.55, True, 9)
]

schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("employee_name", StringType(), False),
    StructField("department", StringType(), False),
    StructField("salary", IntegerType(), False),
    StructField("hire_date", StringType(), False),
    StructField("performance_score", DoubleType(), False),
    StructField("is_active", BooleanType(), False),
    StructField("bonus_percentage", IntegerType(), False)
])

# Create Spark DataFrame
df_spark = spark.createDataFrame(data, schema=schema)

# Convert hire_date to timestamp
df_spark = df_spark.withColumn("hire_date", F.to_timestamp("hire_date", "yyyy-MM-dd"))

print("Sample Dataset Created:")
print("=" * 80)
df_spark.show(10, truncate=False)
print("\n")
print(f"Dataset shape: ({df_spark.count()} rows, {len(df_spark.columns)} columns)")
print(f"\nData types:")
df_spark.printSchema()

## Step 2: Write Data to Avro Format

**About Avro Writing with PySpark:**
- Spark's Avro writer embeds schema information in the file
- Supports multiple compression codecs (SNAPPY, DEFLATE, BZIP2)
- Row-oriented format optimized for write-heavy workloads
- Self-describing format - each file contains its schema

In [ ]:
# Define the file path
avro_file_path = '/Users/mukesh/Desktop/Trainings/nodeB/dataeng/jan_2026/File_Formats/employees_avro'

# Write the DataFrame to Avro format
print("Writing data to Avro format...")
df_spark.coalesce(1).write \
    .mode("overwrite") \
    .format("avro") \
    .option("compression", "snappy") \
    .save(avro_file_path)

print(f"✓ Avro file created successfully at: {avro_file_path}")

# Get file size
total_size = 0
for root, dirs, files in os.walk(avro_file_path):
    for file in files:
        if file.endswith('.avro'):
            file_path = os.path.join(root, file)
            total_size += os.path.getsize(file_path)

print(f"✓ File size: {total_size / 1024:.2f} KB")
print(f"✓ Compression: SNAPPY")
print(f"✓ Format: Avro (Row-Based Storage)")

## Step 3: Read Avro File and Verify Data

**Reading Avro Files with PySpark:**
- Schema is embedded in the file, no need for external schema definition
- Automatic decompression and schema resolution
- Fast deserialization for row-based access
- Supports schema evolution

In [ ]:
# Read the Avro file back
print("Reading Avro file...")
df_read_spark = spark.read.format("avro").load(avro_file_path)

print("Data read from Avro file:")
print("=" * 80)
df_read_spark.show(10, truncate=False)
print("\n")

# Verify data integrity
print("Data Integrity Check:")
print("=" * 80)
print(f"Original rows: {df_spark.count()}")
print(f"Read rows: {df_read_spark.count()}")
print(f"Data matches: {df_spark.count() == df_read_spark.count()}")

# Compare schemas
print("\nSchema comparison:")
print("-" * 80)
print("Original Schema:")
df_spark.printSchema()
print("\nRead Schema:")
df_read_spark.printSchema()

## Step 4: Test 1 - Schema and Metadata Inspection

**Avro Schema:**
- Schema is stored in JSON format within the Avro file
- Self-describing format - no external schema files needed
- Supports complex data types (records, arrays, maps, unions)
- Schema evolution allows backward/forward compatibility

In [ ]:
print("TEST 1: Schema and Metadata Inspection")
print("=" * 80)

# Get schema information
schema = df_read_spark.schema
print("\nSchema Information:")
print("-" * 80)
df_read_spark.printSchema()

# Get column information
print("\nColumn Details:")
print("-" * 80)
for i, field in enumerate(schema.fields):
    print(f"{i+1}. Column: {field.name}")
    print(f"   Type: {field.dataType}")
    print(f"   Nullable: {field.nullable}")
    print()

# Number of rows and columns
row_count = df_read_spark.count()
column_count = len(df_read_spark.columns)
print(f"Total rows: {row_count}")
print(f"Total columns: {column_count}")
print(f"Column names: {', '.join(df_read_spark.columns)}")

## Step 5: Test 2 - Row Access and Filtering

**Theory:**
- Avro's row-based format is optimized for full-row reads
- Efficient for transactional workloads and streaming
- Better write performance compared to columnar formats
- Good for scenarios where all columns are typically accessed

In [ ]:
print("TEST 2: Row Access and Filtering")
print("=" * 80)

# Test 1: Read specific rows
print(f"\nReading specific rows (employee_id > 105):")
print("-" * 80)

filtered_df = df_read_spark.filter(F.col('employee_id') > 105)
filtered_df.show(10, truncate=False)

filtered_count = filtered_df.count()
original_count = df_read_spark.count()
print(f"\nFiltered rows: {filtered_count} out of {original_count}")
print(f"Reduction: {(1 - filtered_count / original_count) * 100:.1f}%")

# Test 2: Multiple filters
print("\n" + "=" * 80)
print("Reading with multiple filters (salary > 70000 AND is_active = true)")
print("-" * 80)

multi_filter_df = df_read_spark.filter(
    (F.col('salary') > 70000) & (F.col('is_active') == True)
)
multi_filter_df.show(10, truncate=False)

print(f"\nFiltered rows: {multi_filter_df.count()} out of {original_count}")

## Step 6: Test 3 - Compression and Format Comparison

**Compression in Avro:**
- Avro supports SNAPPY, DEFLATE, BZIP2, and uncompressed
- SNAPPY provides good balance of speed and compression
- Block-level compression allows random access
- Typically smaller than JSON/CSV, larger than columnar formats for analytics

In [ ]:
print("TEST 3: Compression and Format Comparison")
print("=" * 80)

# Get Avro file size
avro_total_size = 0
for root, dirs, files in os.walk(avro_file_path):
    for file in files:
        if file.endswith('.avro'):
            file_path = os.path.join(root, file)
            avro_total_size += os.path.getsize(file_path)

print(f"\nAvro Format (SNAPPY Compression):")
print(f"  File size: {avro_total_size / 1024:.2f} KB")

# Write as CSV for comparison
csv_file_path = '/Users/mukesh/Desktop/Trainings/nodeB/dataeng/jan_2026/File_Formats/employees_avro.csv'
df_read_spark.coalesce(1).write.mode("overwrite").format("csv").option("header", "true").save(csv_file_path)

csv_total_size = 0
for root, dirs, files in os.walk(csv_file_path):
    for file in files:
        if file.endswith('.csv'):
            file_path = os.path.join(root, file)
            csv_total_size += os.path.getsize(file_path)

print(f"\nCSV Format (uncompressed):")
print(f"  File size: {csv_total_size / 1024:.2f} KB")

# Write as Parquet for comparison
parquet_file_path = '/Users/mukesh/Desktop/Trainings/nodeB/dataeng/jan_2026/File_Formats/employees_avro.parquet'
df_read_spark.coalesce(1).write.mode("overwrite").format("parquet").save(parquet_file_path)

parquet_total_size = 0
for root, dirs, files in os.walk(parquet_file_path):
    for file in files:
        if file.endswith('.parquet'):
            file_path = os.path.join(root, file)
            parquet_total_size += os.path.getsize(file_path)

print(f"\nParquet Format (SNAPPY Compression):")
print(f"  File size: {parquet_total_size / 1024:.2f} KB")

# Comparison summary
print("\n" + "=" * 80)
print("Format Comparison:")
print("-" * 80)
print(f"CSV (baseline):     {csv_total_size / 1024:.2f} KB (100%)")
print(f"Avro (SNAPPY):      {avro_total_size / 1024:.2f} KB ({avro_total_size / csv_total_size * 100:.1f}% of CSV)")
print(f"Parquet (SNAPPY):   {parquet_total_size / 1024:.2f} KB ({parquet_total_size / csv_total_size * 100:.1f}% of CSV)")

print("\n" + "=" * 80)
print("Space Saved vs CSV:")
print("-" * 80)
print(f"Avro:      Save {(1 - avro_total_size / csv_total_size) * 100:.1f}%")
print(f"Parquet:   Save {(1 - parquet_total_size / csv_total_size) * 100:.1f}%")

## Step 7: Test 4 - Data Statistics and Aggregations

In [ ]:
print("TEST 4: Data Statistics and Aggregations")
print("=" * 80)

# Perform various aggregations using Spark SQL
print("\nNumeric Column Statistics:")
print("-" * 80)

numeric_cols = ['salary', 'performance_score', 'bonus_percentage']
for col in numeric_cols:
    stats = df_read_spark.agg(
        F.count(col).alias(f"{col}_count"),
        F.mean(col).alias(f"{col}_mean"),
        F.min(col).alias(f"{col}_min"),
        F.max(col).alias(f"{col}_max"),
        F.stddev(col).alias(f"{col}_stddev")
    ).collect()[0]
    
    print(f"\n{col}:")
    print(f"  Count:   {int(stats[f'{col}_count'])}")
    print(f"  Mean:    {stats[f'{col}_mean']:.2f}")
    print(f"  Min:     {stats[f'{col}_min']}")
    print(f"  Max:     {stats[f'{col}_max']}")
    print(f"  Std Dev: {stats[f'{col}_stddev']:.2f}")

# Department-wise analysis
print("\n" + "=" * 80)
print("Department-wise Analysis:")
print("-" * 80)

dept_analysis = df_read_spark.groupBy('department').agg(
    F.count('*').alias('count'),
    F.mean('salary').alias('avg_salary'),
    F.min('salary').alias('min_salary'),
    F.max('salary').alias('max_salary'),
    F.mean('performance_score').alias('avg_performance')
).orderBy('department')

dept_analysis.show(10, truncate=False)

# Boolean column analysis
print("\n" + "=" * 80)
print("Boolean Column Analysis:")
print("-" * 80)

active_count = df_read_spark.filter(F.col('is_active') == True).count()
inactive_count = df_read_spark.filter(F.col('is_active') == False).count()
total_count = df_read_spark.count()

print(f"Active employees: {active_count}")
print(f"Inactive employees: {inactive_count}")
print(f"Activity rate: {active_count / total_count * 100:.1f}%")

## Step 8: Test 5 - Schema Evolution Demo

**Schema Evolution in Avro:**
- Avro's key strength - supports backward and forward compatibility
- Readers can use different schema version than writers
- Default values allow adding new fields
- Aliases allow renaming fields
- Critical for long-lived data pipelines

In [ ]:
print("TEST 5: Schema Evolution Demonstration")
print("=" * 80)

# Create a new DataFrame with an additional column
print("\nAdding new column 'email' to demonstrate schema evolution...")
print("-" * 80)

df_evolved = df_spark.withColumn(
    "email",
    F.concat(
        F.lower(F.regexp_replace(F.col("employee_name"), " ", ".")),
        F.lit("@company.com")
    )
)

print("\nNew DataFrame with evolved schema:")
df_evolved.show(5, truncate=False)

print("\nOriginal Schema:")
print("-" * 80)
df_spark.printSchema()

print("\nEvolved Schema (with email):")
print("-" * 80)
df_evolved.printSchema()

# Write evolved schema to a new file
evolved_path = '/Users/mukesh/Desktop/Trainings/nodeB/dataeng/jan_2026/File_Formats/employees_avro_evolved'
df_evolved.coalesce(1).write.mode("overwrite").format("avro").save(evolved_path)

print("\n✓ Evolved schema written successfully")
print("\nNote: Avro supports reading old files with new schema and vice versa,")
print("allowing seamless schema evolution in production systems.")

## Step 9: Test 6 - Write Performance Comparison

**Avro Write Performance:**
- Row-based format optimized for fast writes
- Better write performance than columnar formats (ORC/Parquet)
- Ideal for streaming and high-throughput ingestion
- Common in Kafka and other messaging systems

In [ ]:
import time

print("TEST 6: Write Performance Comparison")
print("=" * 80)

# Measure Avro write time
print("\nMeasuring write performance...")
print("-" * 80)

test_path_avro = '/Users/mukesh/Desktop/Trainings/nodeB/dataeng/jan_2026/File_Formats/perf_test_avro'
start_time = time.time()
df_spark.coalesce(1).write.mode("overwrite").format("avro").save(test_path_avro)
avro_write_time = time.time() - start_time

print(f"Avro write time: {avro_write_time:.4f} seconds")

# Measure Parquet write time
test_path_parquet = '/Users/mukesh/Desktop/Trainings/nodeB/dataeng/jan_2026/File_Formats/perf_test_parquet'
start_time = time.time()
df_spark.coalesce(1).write.mode("overwrite").format("parquet").save(test_path_parquet)
parquet_write_time = time.time() - start_time

print(f"Parquet write time: {parquet_write_time:.4f} seconds")

# Measure CSV write time
test_path_csv = '/Users/mukesh/Desktop/Trainings/nodeB/dataeng/jan_2026/File_Formats/perf_test_csv'
start_time = time.time()
df_spark.coalesce(1).write.mode("overwrite").format("csv").option("header", "true").save(test_path_csv)
csv_write_time = time.time() - start_time

print(f"CSV write time: {csv_write_time:.4f} seconds")

print("\n" + "=" * 80)
print("Write Performance Summary:")
print("-" * 80)
print(f"CSV:     {csv_write_time:.4f}s (baseline)")
print(f"Avro:    {avro_write_time:.4f}s ({avro_write_time/csv_write_time*100:.1f}% of CSV)")
print(f"Parquet: {parquet_write_time:.4f}s ({parquet_write_time/csv_write_time*100:.1f}% of CSV)")

print("\nNote: Avro typically shows good write performance due to its row-based structure.")

## Step 10: Test 7 - Complete Format Comparison

**Avro vs Other Formats:**
- **Avro** vs CSV: Binary format, schema embedded, smaller size, type-safe
- **Avro** vs JSON: Compact binary vs text, faster serialization
- **Avro** vs Parquet: Row-based vs columnar, better for writes vs better for reads
- **Avro** vs ORC: Similar trade-offs, Avro more common in streaming

In [ ]:
print("TEST 7: Complete Format Comparison")
print("=" * 80)

# Get sizes for all formats
csv_total_size = 0
for root, dirs, files in os.walk(csv_file_path):
    for file in files:
        if file.endswith('.csv'):
            csv_total_size += os.path.getsize(os.path.join(root, file))

parquet_total_size = 0
for root, dirs, files in os.walk(parquet_file_path):
    for file in files:
        if file.endswith('.parquet'):
            parquet_total_size += os.path.getsize(os.path.join(root, file))

avro_total_size = 0
for root, dirs, files in os.walk(avro_file_path):
    for file in files:
        if file.endswith('.avro'):
            avro_total_size += os.path.getsize(os.path.join(root, file))

# Write as JSON for comparison
json_file = '/Users/mukesh/Desktop/Trainings/nodeB/dataeng/jan_2026/File_Formats/employees_avro.json'
df_read_spark.coalesce(1).write.mode("overwrite").format("json").save(json_file)

json_total_size = 0
for root, dirs, files in os.walk(json_file):
    for file in files:
        if file.endswith('.json'):
            json_total_size += os.path.getsize(os.path.join(root, file))

# Build comparison table
comparison_data = {
    'Format': ['CSV', 'JSON', 'Parquet', 'Avro'],
    'File Size (KB)': [
        f"{csv_total_size / 1024:.2f}",
        f"{json_total_size / 1024:.2f}",
        f"{parquet_total_size / 1024:.2f}",
        f"{avro_total_size / 1024:.2f}"
    ],
    'Compression': [
        'None',
        'None',
        'SNAPPY',
        'SNAPPY'
    ],
    'Type Safety': [
        'No',
        'No',
        'Yes',
        'Yes'
    ],
    'Storage': [
        'Row',
        'Row',
        'Columnar',
        'Row'
    ],
    'Schema Evolution': [
        'No',
        'No',
        'Limited',
        'Yes'
    ]
}

comparison_df = pd.DataFrame(comparison_data)

print("\nFormat Comparison (Using PySpark):")
print("-" * 80)
print(comparison_df.to_string(index=False))

# Calculate compression ratios
print("\n" + "=" * 80)
print("Compression Ratios (vs CSV baseline):")
print("-" * 80)

csv_baseline = csv_total_size
print(f"CSV:      {csv_baseline / 1024:.2f} KB (100%)")
print(f"JSON:     {json_total_size / 1024:.2f} KB ({json_total_size / csv_baseline * 100:.1f}%)")
print(f"Parquet:  {parquet_total_size / 1024:.2f} KB ({parquet_total_size / csv_baseline * 100:.1f}%)")
print(f"Avro:     {avro_total_size / 1024:.2f} KB ({avro_total_size / csv_baseline * 100:.1f}%)")

print("\n" + "=" * 80)
print("Space Saved vs CSV:")
print("-" * 80)
print(f"JSON:      Save {max(0, (1 - json_total_size / csv_baseline) * 100):.1f}%")
print(f"Parquet:   Save {(1 - parquet_total_size / csv_baseline) * 100:.1f}%")
print(f"Avro:      Save {(1 - avro_total_size / csv_baseline) * 100:.1f}%")

## Summary: Avro Format Benefits and Use Cases with PySpark

### Advantages of Avro Format:
1. **Fast Serialization** - Efficient binary encoding for quick writes
2. **Schema Evolution** - Backward and forward compatibility built-in
3. **Self-Describing** - Schema embedded in file, no external dependencies
4. **Language Neutral** - Works across multiple programming languages
5. **Compact Binary Format** - Smaller than JSON/XML, more efficient than text
6. **Row-Based Storage** - Optimized for transactional and streaming workloads
7. **Good Compression** - Supports multiple codecs with block-level compression
8. **Splittable** - Can be processed in parallel across distributed systems

### PySpark-Specific Advantages:
- **Native Support** - Built-in Avro reader/writer with spark-avro package
- **Streaming Integration** - Perfect for Spark Structured Streaming
- **Kafka Compatibility** - Standard format for Kafka message serialization
- **Schema Registry** - Integrates well with Confluent Schema Registry
- **Fast Writes** - Row-based format enables high-throughput ingestion

### When to Use Avro with PySpark:
- **Event Streaming** - Kafka topics, event logs, real-time data pipelines
- **Messaging Systems** - RPC frameworks, service-to-service communication
- **Data Exchange** - Sharing data between different systems/languages
- **Schema Evolution** - Long-lived data with changing schemas
- **Write-Heavy Workloads** - High-throughput data ingestion
- **ETL Intermediate Format** - Data transformation pipelines
- **Audit Logs** - Immutable event logs with schema evolution

### When NOT to Use Avro:
- **Analytical Queries** - Use Parquet/ORC for column-based analytics
- **Human Readability** - Binary format, not human-readable (use JSON/CSV)
- **Random Column Access** - Columnar formats better for selective columns
- **Maximum Compression** - Columnar formats achieve better compression ratios

### Avro vs. Other Formats:

**Avro vs. Parquet:**
- Avro: Row-based, better for writes, schema evolution
- Parquet: Columnar, better for analytical reads, better compression

**Avro vs. ORC:**
- Avro: Better schema evolution, more common in streaming
- ORC: Better for Hive, optimized for HDFS

**Avro vs. JSON:**
- Avro: Binary, compact, type-safe, schema required
- JSON: Text, human-readable, flexible, no schema

### Key Statistics:
- **Compression Ratio**: 30-60% reduction vs uncompressed
- **Write Performance**: Often faster than columnar formats
- **Schema Evolution**: Full backward/forward compatibility
- **Serialization Speed**: Very fast binary encoding/decoding
- **Common Use Case**: Kafka message serialization (industry standard)

### Best Practices:
1. Use Avro for event streams and messaging
2. Leverage schema evolution for long-lived data
3. Consider Parquet/ORC for analytical workloads
4. Use Schema Registry in production Kafka environments
5. Choose SNAPPY compression for balanced performance
6. Test write vs. read performance for your use case
7. Document schema changes for backward compatibility